# Experiment 12: diverse-model mixture of experts with block → town fallback

Four full-dataset experts—LightGBM, CatBoost, ExtraTrees, and trend-residual LightGBM—are trained from chronological folds. A neural soft gate learns a convex blend from their OOF predictions. Its primary location input is `BLOCK`; unseen blocks map to an unknown embedding while `TOWN` remains available as the fallback signal.

**Executed result:** CatBoost scored **470.363 RMSE** and ExtraTrees **493.016**, but the LightGBM expert was unstable at **2291.784** and the trend-residual expert scored **716.098**. The equal blend consequently failed at **651.703**, while the gate collapsed to 100% weight on the trend-residual expert and also scored **716.098**. The run shows that a gate cannot rescue failed or differently calibrated experts; expert validation must precede gating.

In [1]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from autogluon.tabular import TabularPredictor
from scipy.spatial import cKDTree
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit

configured_root=os.environ.get('KDDM_PROJECT_ROOT'); roots=([Path(configured_root)] if configured_root else [])+[Path.cwd(),*Path.cwd().parents]
project_root=next((p for p in roots if (p/'exploration/metul/preprocessing.py').exists()),None)
if project_root is None: raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir=project_root/'exploration'/'metul'; sys.path.insert(0,str(experiment_dir))
from preprocessing import RentalPreprocessor, load_datasets
TARGET='MONTHLY_RENT'; GROUP=['TOWN','FLAT_TYPE']; LAG_MONTHS=24; RUN_ID=os.environ.get('AUTOGLUON_RUN_ID',pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
BASE_TIME_LIMIT=int(os.environ.get('MOE_LGBM_TIME_LIMIT','240')); N_SPLITS=int(os.environ.get('MOE_OOF_SPLITS','3')); torch.manual_seed(42); np.random.seed(42)
compact=['TOWN','BLOCK','STREET','FLAT_TYPE','FLAT_MODEL','FLOOR_AREA_SQM','approval_date_num','approval_month','coe_quota_premium_mean','YEAR_COMPLETED','MAX_FLOOR','mrt_dist_nearest_km','mall_dist_nearest_km']
FEATURES=compact+['monthly_target_lag','town_flat_type_target_mean','nearest_mrt_name','nearest_tourist_attraction_name','nearest_monument_name']
EXPERTS=['lightgbm','catboost','extra_trees','trend_residual_lgbm']
_,datasets=load_datasets(project_root/'data'); raw_train=datasets['train'].copy(); raw_train['period']=pd.to_datetime(raw_train.RENT_APPROVAL_DATE+'-01').dt.to_period('M'); periods=np.sort(raw_train.period.unique()); cutoff=periods[int(len(periods)*.80)]; raw_fit=raw_train[raw_train.period<cutoff].copy(); raw_valid=raw_train[raw_train.period>=cutoff].copy(); print(f'Outer cutoff={cutoff}; fit={len(raw_fit):,}; validation={len(raw_valid):,}; experts={list(EXPERTS)}')


Outer cutoff=2024-05; fit=117,988; validation=32,012; experts=['lightgbm', 'catboost', 'extra_trees', 'trend_residual_lgbm']


In [2]:
class TargetFeatures:
    def fit(self,h): self.g=h[TARGET].mean(); self.gm=h.groupby(GROUP)[TARGET].mean(); self.mm=h.groupby('period')[TARGET].mean(); return self
    def transform(self,h):
        z=h[GROUP+['period']].copy(); z['monthly_target_lag']=(z.period-LAG_MONTHS).map(self.mm).fillna(self.g); z['town_flat_type_target_mean']=pd.Series(z.set_index(GROUP).index.map(self.gm),index=z.index).fillna(self.g); return z[['monthly_target_lag','town_flat_type_target_mean']]
    def fit_train(self,h):
        self.fit(h); a=h.groupby(GROUP+['period'],as_index=False)[TARGET].agg(['sum','count']).reset_index().sort_values(GROUP+['period']); a['ps']=a.groupby(GROUP)['sum'].cumsum()-a['sum']; a['pc']=a.groupby(GROUP)['count'].cumsum()-a['count']; a['town_flat_type_target_mean']=a.ps/a.pc.replace(0,np.nan); m=h.groupby('period')[TARGET].agg(['sum','count']).sort_index(); prior=(m['sum'].cumsum()-m['sum'])/(m['count'].cumsum()-m['count']).replace(0,np.nan); z=h[GROUP+['period']].merge(a[GROUP+['period','town_flat_type_target_mean']],on=GROUP+['period'],how='left'); z['town_flat_type_target_mean']=z.town_flat_type_target_mean.fillna(z.period.map(prior)); z['monthly_target_lag']=(z.period-LAG_MONTHS).map(self.mm).fillna(z.period.map(prior)); return z[['monthly_target_lag','town_flat_type_target_mean']]
def prep(p,r,fit=False): return p.fit_transform(r.drop(columns=['period',TARGET],errors='ignore')) if fit else p.transform(r.drop(columns=['period',TARGET],errors='ignore'))
def hist(r,b):
    z=b[GROUP].reset_index(drop=True).copy(); z['period']=r.period.to_numpy(); z[TARGET]=r[TARGET].to_numpy(); return z
def mrt(b):
    s=datasets['mrt'][['NAME','LATITUDE','LONGITUDE']].dropna(); _,i=cKDTree(s[['LATITUDE','LONGITUDE']].to_numpy()*[111,110.96]).query(b[['LATITUDE','LONGITUDE']].to_numpy()*[111,110.96]); return pd.Series(s.NAME.astype(object).to_numpy()[i],name='nearest_mrt_name')
def geo(b,file,key,name):
    with (project_root/'data'/'possible_additional_features'/file).open() as h: d=json.load(h); q=pd.DataFrame([{'longitude':x['geometry']['coordinates'][0],'latitude':x['geometry']['coordinates'][1],name:x['properties'][key]} for x in d['features'] if x.get('geometry',{}).get('type')=='Point']).dropna(); _,i=cKDTree(q[['longitude','latitude']].to_numpy()*[110.96,111]).query(b[['LONGITUDE','LATITUDE']].to_numpy()*[110.96,111]); return pd.Series(q[name].astype(object).to_numpy()[i],name=name)
def make_frames(early,later):
    p=RentalPreprocessor(datasets); eb,lb=prep(p,early,True),prep(p,later); t=TargetFeatures(); et,lt=t.fit_train(hist(early,eb)),t.transform(hist(later,lb)); f=lambda b,x:pd.concat([b[compact].reset_index(drop=True),x.reset_index(drop=True),mrt(b).reset_index(drop=True),geo(b,'TouristAttractions.geojson','PAGETITLE','nearest_tourist_attraction_name').reset_index(drop=True),geo(b,'MonumentsGEOJSON.geojson','NAME','nearest_monument_name').reset_index(drop=True)],axis=1)[FEATURES]; return f(eb,et),f(lb,lt)


In [3]:
def fit_ag(x,y,name,model_type,params):
    predictor=TabularPredictor(label=TARGET,problem_type='regression',eval_metric='root_mean_squared_error',path=str(experiment_dir/'ag_exp_12_diverse_moe'/RUN_ID/name/model_type)).fit(train_data=x.assign(**{TARGET:y.to_numpy()}),hyperparameters={model_type:params},fit_weighted_ensemble=False,presets=['medium_quality','ignore_text'],time_limit=BASE_TIME_LIMIT,verbosity=2)
    return predictor,next(model for model in predictor.model_names() if model.startswith('LightGBM' if model_type=='GBM' else 'CatBoost'))
def make_extra(x):
    cats=x.select_dtypes(include=['object','category','string']).columns.tolist(); nums=[c for c in FEATURES if c not in cats]
    return Pipeline([('encode',ColumnTransformer([('n',SimpleImputer(strategy='median'),nums),('c',Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]),cats)])),('model',ExtraTreesRegressor(n_estimators=400,min_samples_leaf=3,max_features=.7,n_jobs=-1,random_state=42))])
def fit_experts(x,y,name):
    lgb=fit_ag(x,y,name,'GBM',{'num_leaves':63,'learning_rate':.04,'num_boost_round':1800,'min_data_in_leaf':25,'feature_fraction':.9})
    cat=fit_ag(x,y,name,'CAT',{'iterations':1800,'depth':8,'learning_rate':.04,'l2_leaf_reg':5.0})
    extra=make_extra(x).fit(x,y)
    trend=x.monthly_target_lag.fillna(y.mean()); residual_lgb=fit_ag(x,y-trend,name,'GBM',{'num_leaves':31,'learning_rate':.03,'num_boost_round':1400,'min_data_in_leaf':40,'lambda_l2':5.0})
    return {'lightgbm':('ag',*lgb),'catboost':('ag',*cat),'extra_trees':('sk',extra),'trend_residual_lgbm':('trend',*residual_lgb)}
def predict_expert(info,x):
    if info[0]=='sk': return info[1].predict(x).astype(float)
    prediction=info[1].predict(x,model=info[2]).astype(float).to_numpy()
    return prediction if info[0]=='ag' else prediction+x.monthly_target_lag.fillna(x.monthly_target_lag.median()).to_numpy()
fit_periods=np.sort(raw_fit.period.unique()); parts=[]
for fold,(tr,va) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods),1):
    early=raw_fit[raw_fit.period.isin(fit_periods[tr])].copy(); later=raw_fit[raw_fit.period.isin(fit_periods[va])].copy(); xtr,xva=make_frames(early,later); models=fit_experts(xtr,early[TARGET].astype(float).reset_index(drop=True),f'oof_{fold}'); out=xva[['BLOCK','TOWN']].reset_index(drop=True).copy(); out['target']=later[TARGET].to_numpy(); out['fold']=fold
    for expert,info in models.items(): out[expert]=predict_expert(info,xva)
    parts.append(out); print(f'OOF fold {fold} complete: {len(out):,} rows')
oof=pd.concat(parts,ignore_index=True); assert oof[list(EXPERTS)].notna().all().all()


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.81 GB / 16.00 GB (36.3%)
Disk Space Avail:   176.50 GB / 460.38 GB (38.3%)
Presets specified: ['medium_quality', 'ignore_text']


[categorical_clean] derived HDB feature | REGION: 5 -> 5 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Beginning AutoGluon training ... Time limit = 240s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_12_diverse_moe/20260929_202859/oof_1/GBM"
Train Data Rows:    31565
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    5963.05 MB
	Train Data (Original)  Memory Usage: 18.30 MB (0.3% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	St

OOF fold 1 complete: 28,514 rows
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/undersc

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.65 GB / 16.00 GB (35.3%)
Disk Space Avail:   176.48 GB / 460.38 GB (38.3%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 240s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_12_diverse_moe/20260929_202859/oof_2/GBM"
Train Data Rows:    60079
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to pre

OOF fold 2 complete: 27,858 rows
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/undersc

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.80 GB / 16.00 GB (30.0%)
Disk Space Avail:   176.45 GB / 460.38 GB (38.3%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 240s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_12_diverse_moe/20260929_202859/oof_3/GBM"
Train Data Rows:    87937
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to pre

OOF fold 3 complete: 30,051 rows


In [4]:
# Vocabularies are fitted only on OOF gate-training rows. Index 0 is unknown: unseen blocks therefore fall back to town context.
stop_fold=oof.fold.max(); gate_train=oof[oof.fold<stop_fold].copy(); gate_stop=oof[oof.fold==stop_fold].copy()
def vocab(values): return {value:i+1 for i,value in enumerate(sorted(values.dropna().astype(str).unique()))}
block_vocab,town_vocab=vocab(gate_train.BLOCK),vocab(gate_train.TOWN)
def indices(frame,column,mapping): return torch.tensor(frame[column].astype(str).map(mapping).fillna(0).astype(int).to_numpy(),dtype=torch.long)
class BlockTownGate(nn.Module):
    def __init__(self):
        super().__init__(); self.block=nn.Embedding(len(block_vocab)+1,16); self.town=nn.Embedding(len(town_vocab)+1,8); self.net=nn.Sequential(nn.Linear(16+8+len(EXPERTS),48),nn.ReLU(),nn.Dropout(.1),nn.Linear(48,len(EXPERTS)))
    def forward(self,b,t,p):
        w=torch.softmax(self.net(torch.cat([self.block(b),self.town(t),p],1)),1); return (w*p).sum(1),w
def tensor(frame): return torch.tensor(frame[list(EXPERTS)].to_numpy(),dtype=torch.float32)
gate=BlockTownGate(); opt=torch.optim.AdamW(gate.parameters(),lr=.001,weight_decay=.0001); loader=DataLoader(TensorDataset(indices(gate_train,'BLOCK',block_vocab),indices(gate_train,'TOWN',town_vocab),tensor(gate_train),torch.tensor(gate_train.target.to_numpy(),dtype=torch.float32)),batch_size=1024,shuffle=True); best,best_loss,stale=None,np.inf,0
for epoch in range(300):
    gate.train()
    for b,t,p,y in loader:
        pred,_=gate(b,t,p); loss=((pred-y)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    gate.eval();
    with torch.no_grad(): sp,_=gate(indices(gate_stop,'BLOCK',block_vocab),indices(gate_stop,'TOWN',town_vocab),tensor(gate_stop)); val=((sp-torch.tensor(gate_stop.target.to_numpy(),dtype=torch.float32))**2).mean().item()
    if val<best_loss-1e-3: best_loss,best,stale=val,{k:v.detach().clone() for k,v in gate.state_dict().items()},0
    else:
        stale+=1
        if stale>=30: break
gate.load_state_dict(best)
xfit,xvalid=make_frames(raw_fit,raw_valid); final=fit_experts(xfit,raw_fit[TARGET].astype(float).reset_index(drop=True),'final'); hold=xvalid[['BLOCK','TOWN']].reset_index(drop=True).copy()
for expert,info in final.items(): hold[expert]=predict_expert(info,xvalid)
with torch.no_grad(): pred,weights=gate(indices(hold,'BLOCK',block_vocab),indices(hold,'TOWN',town_vocab),tensor(hold))
actual=raw_valid[TARGET].astype(float).reset_index(drop=True); hold['gated']=pred.numpy()
for i,expert in enumerate(EXPERTS): hold[f'{expert}_weight']=weights[:,i].numpy()
comparison=pd.DataFrame([{'model':e,'RMSE':mean_squared_error(actual,hold[e])**.5} for e in EXPERTS]+[{'model':'equal blend','RMSE':mean_squared_error(actual,hold[list(EXPERTS)].mean(1))**.5},{'model':'block→town soft gate','RMSE':mean_squared_error(actual,hold.gated)**.5}]); display(comparison.sort_values('RMSE')); print(f'Gate epochs={epoch+1}; unseen validation blocks={(~hold.BLOCK.astype(str).isin(block_vocab)).mean():.1%}')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.65 GB / 16.00 GB (29.0%)
Disk Space Avail:   176.40 GB / 460.38 GB (38.3%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 240s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_12_diverse_moe/20260929_202859/final/GBM"
Train Data Rows:    117988
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to pr

,model,RMSE
1,catboost,470.362927
2,extra_trees,493.015920
4,equal blend,651.702515
3,trend_residual_lgbm,716.098332
5,block→town soft gate,716.098332
0,lightgbm,2291.784333


Gate epochs=31; unseen validation blocks=0.5%


## Gate-weight diagnostics

A healthy mixture should show non-trivial variation across rows and locations. A near-100% share for one expert means the gate has collapsed to expert selection rather than adaptive blending.

In [5]:
weight_columns=[f'{expert}_weight' for expert in EXPERTS]
weight_summary=hold[weight_columns].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
weight_summary['rows_weight_gt_0_90']=(hold[weight_columns]>0.90).sum(); weight_summary['rows_weight_gt_0_50']=(hold[weight_columns]>0.50).sum()
display(weight_summary)
hold['selected_expert']=hold[weight_columns].idxmax(axis=1).str.removesuffix('_weight')
display(hold.selected_expert.value_counts(normalize=True).mul(100).rename('share_percent').to_frame())
hold['weight_entropy']=-(hold[weight_columns]*np.log(hold[weight_columns].clip(lower=1e-12))).sum(axis=1)
print(f'Mean gate entropy: {hold.weight_entropy.mean():.4f} nats (maximum for {len(EXPERTS)} experts: {np.log(len(EXPERTS)):.4f}).')
town_weights=hold.groupby('TOWN')[weight_columns].mean(); town_weights['dominant_expert']=town_weights.idxmax(axis=1).str.removesuffix('_weight'); display(town_weights.sort_values(weight_columns))
block_weights=hold.groupby(['TOWN','BLOCK'])[weight_columns].mean(); block_weights['dominant_expert']=block_weights.idxmax(axis=1).str.removesuffix('_weight'); display(block_weights.sort_values(weight_columns).head(20))
if hold[weight_columns].max(axis=1).mean()>0.98: print('Interpretation: the gate has effectively collapsed to one expert. Its RMSE will closely match that expert, as in the current result.')
else: print('Interpretation: the gate is genuinely varying its blend by location.')


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,rows_weight_gt_0_90,rows_weight_gt_0_50
lightgbm_weight,32012.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
catboost_weight,32012.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
extra_trees_weight,32012.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
trend_residual_lgbm_weight,32012.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,32012,32012


,share_percent
selected_expert,
trend_residual_lgbm,100.0


Mean gate entropy: 0.0000 nats (maximum for 4 experts: 1.3863).


,lightgbm_weight,catboost_weight,extra_trees_weight,trend_residual_lgbm_weight,dominant_expert
TOWN,,,,,
ang mo kio,0.0,0.0,0.0,1.0,trend_residual_lgbm
bedok,0.0,0.0,0.0,1.0,trend_residual_lgbm
bishan,0.0,0.0,0.0,1.0,trend_residual_lgbm
bukit batok,0.0,0.0,0.0,1.0,trend_residual_lgbm
bukit merah,0.0,0.0,0.0,1.0,trend_residual_lgbm
bukit panjang,0.0,0.0,0.0,1.0,trend_residual_lgbm
bukit timah,0.0,0.0,0.0,1.0,trend_residual_lgbm
central,0.0,0.0,0.0,1.0,trend_residual_lgbm
choa chu kang,0.0,0.0,0.0,1.0,trend_residual_lgbm


lightgbm_weight  catboost_weight  extra_trees_weight  \
TOWN       BLOCK                                                         
ang mo kio 101                0.0              0.0                 0.0   
           102                0.0              0.0                 0.0   
           103                0.0              0.0                 0.0   
           104b               0.0              0.0                 0.0   
           105                0.0              0.0                 0.0   
           106                0.0              0.0                 0.0   
           107                0.0              0.0                 0.0   
           108                0.0              0.0                 0.0   
           109                0.0              0.0                 0.0   
           110                0.0              0.0                 0.0   
           111                0.0              0.0                 0.0   
           112                0.0              0.0                 0.0   
           113                0.0              0.0                 0.0   
           114                0.0              0.0                 0.0   
           117                0.0              0.0                 0.0   
           118                0.0              0.0                 0.0   
           119                0.0              0.0                 0.0   
           120                0.0              0.0                 0.0   
           121                0.0              0.0                 0.0   
           122                0.0              0.0                 0.0   

                  trend_residual_lgbm_weight      dominant_expert  
TOWN       BLOCK                                                   
ang mo kio 101                           1.0  trend_residual_lgbm  
           102                           1.0  trend_residual_lgbm  
           103                           1.0  trend_residual_lgbm  
           104b                          1.0  trend_residual_lgbm  
           105                           1.0  trend_residual_lgbm  
           106                           1.0  trend_residual_lgbm  
           107                           1.0  trend_residual_lgbm  
           108                           1.0  trend_residual_lgbm  
           109                           1.0  trend_residual_lgbm  
           110                           1.0  trend_residual_lgbm  
           111                           1.0  trend_residual_lgbm  
           112                           1.0  trend_residual_lgbm  
           113                           1.0  trend_residual_lgbm  
           114                           1.0  trend_residual_lgbm  
           117                           1.0  trend_residual_lgbm  
           118                           1.0  trend_residual_lgbm  
           119                           1.0  trend_residual_lgbm  
           120                           1.0  trend_residual_lgbm  
           121                           1.0  trend_residual_lgbm  
           122                           1.0  trend_residual_lgbm

Interpretation: the gate has effectively collapsed to one expert. Its RMSE will closely match that expert, as in the current result.
